[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/int/u4_tfc.ipynb)

# Int. U4 · Teorema fundamental del cálculo
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 4 del bloque de cálculo integral. Tiene una sección por subtema, en el mismo orden que el libro:

1. Primera parte: la derivada de la integral
2. Segunda parte: evaluación con la antiderivada
3. Evaluación de integrales definidas (integral con signo y área total)
4. Teorema del valor medio para integrales
5. Comprobación numérica contra la suma de Riemann

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `INT-U4` (bloque, unidad). Las celdas de código de cada sección vienen de `int/u4_tfc/NN_*.py`; los fragmentos que el libro imprime, de `int/u4_tfc/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir funciones usa `x` como variable, `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)`, `exp(...)`, `log(...)` (natural) y `abs(...)`. Los ángulos de `sin`, `cos`, `tan` y `sec` van en **radianes**. Antes de aplicar $F(b)-F(a)$, las calculadoras revisan que $f$ sea continua en $[a,b]$.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8" "scipy>=1.11"

import sys, math, re
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from scipy.optimize import brentq
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)

import scipy
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| scipy", scipy.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x = sp.symbols("x", real=True)

_FUNCIONES = {"sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log, "ln": sp.log,
              "pi": sp.pi, "sin": sp.sin, "sen": sp.sin, "cos": sp.cos, "tan": sp.tan, "sec": sp.sec,
              "E": sp.E, "e": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
# rationalize: 0.1 se lee como 1/10, para que los resultados salgan exactos
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)
_OTRAS_VARIABLES = ("t", "u", "w", "q", "s")      # se leen como x si son la única variable
_C = sp.Symbol("C")


def parsear(texto, variables=("x",)):
    """Convierte un texto en una expresión de sympy o explica qué salió mal.
    Acepta sen y ln, la letra e como número de Euler, una constante + C (se descarta)
    y otra letra (t, u, w, q o s) en lugar de x si es la única variable."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una función, por ejemplo: x**2")
    nombres = set(re.findall(r"[A-Za-z_]\w*", texto))
    otras = [v for v in _OTRAS_VARIABLES if v in nombres]
    if "x" not in nombres and len(otras) == 1:
        texto = re.sub(rf"(?<![A-Za-z_]){otras[0]}(?![A-Za-z_0-9])", "x", texto)
        nombres = set(re.findall(r"[A-Za-z_]\w*", texto))
    local = {**_FUNCIONES, "x": x, "C": _C}
    desconocidos = sorted(nombres - set(local))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa x (o t) como variable y, si hace falta, "
                         "sqrt, exp, log o ln, abs, sin o sen, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=local, global_dict=dict(_GLOBAL), transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la función. Usa * para multiplicar, ** o ^ para potencias "
                         "y sqrt(...) para raíces.") from err
    if not isinstance(expr, sp.Expr) or expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Escribe una expresión sin el signo = y solo con las funciones permitidas.")
    expr = expr.subs(_C, 0)                       # la constante de integración no cambia la derivada
    if expr.free_symbols - {x}:
        raise ValueError("Usa una sola variable.")
    return expr


def lista_numeros(texto, nombre="la lista"):
    """'0, 1.5, 3' -> [0.0, 1.5, 3.0], o un mensaje claro si algo no es número."""
    partes = [p for p in re.split(r"[,\s;]+", str(texto).strip()) if p]
    if not partes:
        raise ValueError(f"Escribe {nombre} como números separados por comas, por ejemplo: 0, 1, 3, 4")
    try:
        return [float(p) for p in partes]
    except ValueError as err:
        raise ValueError(f"En {nombre} hay algo que no es un número: usa punto decimal y comas entre valores.") from err


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    d = Decimal(repr(v))
    q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() != d.adjusted():          # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
        q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() >= n:                     # enteros grandes: sin notación científica
        return f"{int(q):d}"
    return format(q, "f")


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def valor_real(f, c):
    """f(c) como float, o ValueError si no es un número real finito."""
    v = complex(sp.N(f.subs(x, c)))
    if abs(v.imag) > 1e-12 or not math.isfinite(v.real):
        raise ValueError(f"f no tiene valor real en x = {cifras(float(c), 6)}: revisa el dominio.")
    return v.real


def numerica(f):
    """Versión numpy de f que siempre devuelve un arreglo del tamaño de la entrada."""
    g = sp.lambdify(x, f, "numpy")
    return lambda t: np.asarray(g(np.asarray(t, dtype=float)), dtype=float) * np.ones_like(np.asarray(t, dtype=float))


# Antiderivadas (las mismas herramientas de la Unidad 3)
_PERMITIDAS = (sp.exp, sp.log, sp.sin, sp.cos, sp.tan, sp.sec, sp.Abs, sp.atan, sp.asin, sp.acos)
_PUNTOS = [-2.9, -2.1, -1.3, -0.7, 0.7, 1.3, 2.1, 2.9]


def con_valor_absoluto(F):
    """ln(u) -> ln|u| salvo que u sea siempre positivo: la antiderivada de u'/u vale donde u > 0 y donde u < 0."""
    return F.replace(sp.log, lambda a: sp.log(a) if (a.is_positive or a.has(sp.Abs)) else sp.log(sp.Abs(a)))


def sin_abs_en_log(f):
    """ln|u| -> ln u, solo para integrar (la derivada de las dos es u'/u); el valor absoluto se repone al final."""
    return f.replace(lambda e: isinstance(e, sp.log) and e.args[0].has(sp.Abs),
                     lambda e: sp.log(e.args[0].replace(sp.Abs, lambda a: a)))


def misma_funcion(g, h):
    """True si g y h coinciden: primero con sympy; si no decide, en los puntos donde las dos son reales."""
    d = sp.simplify(g - h)
    if d == 0:
        return True
    comparados = 0
    for c in _PUNTOS:
        try:
            if abs(valor_real(g, c) - valor_real(h, c)) > 1e-9:
                return False
            comparados += 1
        except (ValueError, TypeError, ZeroDivisionError):
            continue
    return comparados >= 3


def antiderivada(f):
    """Una antiderivada de f (con C = 0), simplificada y con ln|..| donde corresponde."""
    F = sp.integrate(sin_abs_en_log(f), x)
    if F.has(sp.Integral):
        raise ValueError("sympy no encontró una antiderivada para esta función.")
    raras = {type(a).__name__ for a in F.atoms(sp.Function) if not isinstance(a, _PERMITIDAS)}
    if raras:
        raise ValueError(f"la antiderivada usa funciones especiales ({', '.join(sorted(raras))}): no es elemental. "
                         "Calcula la integral definida con las sumas de la sección 5 de este notebook.")
    F = sp.expand(F) if F.is_polynomial(x) else con_valor_absoluto(sp.trigsimp(sp.simplify(F)))
    if not misma_funcion(derivada_legible(F), sin_abs_en_log(f)):
        raise ValueError("la comprobación F' = f falló; revisa la función.")
    return F


def derivada_legible(F):
    """F' sin el Piecewise que sympy agrega al derivar ln|u| (la derivada de ln|u| es u'/u)."""
    G = F.replace(lambda e: isinstance(e, sp.log) and e.args[0].has(sp.Abs),
                  lambda e: sp.log(e.args[0].replace(sp.Abs, lambda a: a)))
    return sp.simplify(sp.diff(G, x))


def continua_en(f, a, b):
    """True si f es continua en todo el intervalo cerrado entre a y b."""
    lo, hi = sorted((sp.nsimplify(a), sp.nsimplify(b)))
    dom = sp.calculus.util.continuous_domain(f, x, sp.Interval(lo, hi))
    return dom == sp.Interval(lo, hi)

## 1. Primera parte: la derivada de la integral

Si $f$ es continua en $[a,b]$ y $G(x)=\int_a^x f(t)\,dt$, entonces $G'(x)=f(x)$. Con límites que dependen de $x$:

$$\frac{d}{dx}\int_{u(x)}^{v(x)}f(t)\,dt=f\big(v(x)\big)\,v'(x)-f\big(u(x)\big)\,u'(x).$$

La primera calculadora construye $G$ con trapecios y compara su derivada numérica con $f$. La segunda deriva una integral con límites variables y muestra cada término de la regla de la cadena. Escribe $f$ con la variable `x`; la calculadora la lee como $f(t)$.

In [ ]:
def acumulada(f, a, b, m=2000):
    """(t, G) con G(t) = ∫_a^t f, acumulada con trapecios en m franjas."""
    if not a < b:
        raise ValueError("el intervalo debe cumplir a < b.")
    if not continua_en(f, a, b):
        raise ValueError("f no es continua en todo [a, b]; la primera parte del teorema pide continuidad.")
    F = numerica(f)
    t = np.linspace(a, b, m + 1)
    y = F(t)
    G = np.concatenate([[0.0], np.cumsum((y[:-1] + y[1:]) / 2 * np.diff(t))])
    return t, G


def derivada_integral(f, u, v):
    """d/dx de ∫_{u(x)}^{v(x)} f(t) dt y sus dos términos."""
    alto = f.subs(x, v) * sp.diff(v, x)
    bajo = f.subs(x, u) * sp.diff(u, x)
    return sp.simplify(alto - bajo), sp.simplify(alto), sp.simplify(bajo)


def calculadora_acumulada(f_txt, a, b, n_cifras):
    try:
        f = parsear(f_txt)
        t, G = acumulada(f, a, b)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    dG = np.gradient(G, t[1] - t[0])
    print(f"G({b}) = ∫ de {a} a {b} ≈ {cifras(G[-1], n_cifras)}   ({n_cifras} cifras significativas, redondeado)")
    print("máx |G' - f| en puntos interiores ≈", cifras(np.max(np.abs(dG[1:-1] - numerica(f)(t[1:-1]))), 2))
    fig, ax = plt.subplots(figsize=(5.5, 3))
    ax.plot(t, numerica(f)(t), color="navy", label="f(t)")
    ax.plot(t, G, color="red", label="G(x)")
    ax.axhline(0, color="black", lw=0.6); ax.set_xlabel("x"); ax.legend(); plt.show()


def calculadora_limites(f_txt, u_txt, v_txt):
    try:
        f, u, v = parsear(f_txt), parsear(u_txt), parsear(v_txt)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    total, alto, bajo = derivada_integral(f, u, v)
    if sp.calculus.util.continuous_domain(f, x, sp.S.Reals) != sp.S.Reals:
        print("Aviso: f no es continua en toda la recta; la fórmula vale solo si f es continua entre los dos límites.")
    print(f"d/dx ∫ de {u} a {v} de f(t) dt, con f(t) = {f.subs(x, sp.Symbol('t'))}")
    print("  término del límite superior: f(v)·v' =", alto)
    print("  término del límite inferior: f(u)·u' =", bajo)
    print("  derivada =", total)


widgets.interact(calculadora_acumulada,
    f_txt=widgets.Text(value="2 + cos(x)", description="f ="),
    a=widgets.FloatText(value=0, description="a"), b=widgets.FloatText(value=4, description="b"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=8, description="cifras sig."));
widgets.interact(calculadora_limites,
    f_txt=widgets.Text(value="cos(x)", description="f ="),
    u_txt=widgets.Text(value="0", description="u(x) ="), v_txt=widgets.Text(value="x**2", description="v(x) ="));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
PRUEBAS_1 = [
    ("d/dx ∫_0^x t^2 dt = x^2", lambda: misma_funcion(derivada_integral(x**2, 0 * x, x)[0], x**2)),
    ("d/dx ∫_1^{x^2} dt/t = 2/x", lambda: misma_funcion(derivada_integral(1 / x, 1 + 0 * x, x**2)[0], 2 / x)),
    ("d/dx ∫_x^5 t dt = -x", lambda: misma_funcion(derivada_integral(x, x, 5 + 0 * x)[0], -x)),
    ("G de cos en [0, π]: G(π/2) ≈ 1", lambda: cerca(np.interp(math.pi / 2, *acumulada(sp.cos(x), 0, math.pi)), 1, 1e-6)),
    ("e^(-t^2) en [0, 2]: máx |G' - f| < 1e-4",
     lambda: (lambda t, G: np.max(np.abs(np.gradient(G, t[1] - t[0])[1:-1] - np.exp(-t[1:-1]**2))) < 1e-4)(
         *acumulada(sp.exp(-x**2), 0, 2))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Deriva a mano $H(x)=\int_0^{3x}e^{t^2}\,dt$.

In [ ]:
mi_H = None     # escribe H'(x) como texto, con x como variable

if mi_H is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    ref = derivada_integral(sp.exp(x**2), 0 * x, 3 * x)[0]
    print("La calculadora da:", ref)
    try:
        ok = misma_funcion(parsear(str(mi_H)), ref)
    except ValueError as err:
        print(err); ok = False
    print("coinciden" if ok else "NO coinciden: evalúa e^(t^2) en el límite superior, 3x, y multiplica por la derivada de 3x")

## 2. Segunda parte: evaluación con la antiderivada

$$\int_a^b f(x)\,dx=F(b)-F(a)=\Big[F(x)\Big]_a^b,\qquad\text{si } f \text{ es continua en } [a,b] \text{ y } F'=f.$$

La calculadora revisa primero la continuidad de $f$ en $[a,b]$; si no se cumple, **no** aplica el teorema y explica por qué. Después busca una antiderivada, muestra la resta y la compara con una suma de punto medio con 1000 franjas.

In [ ]:
def evaluar(f, a, b):
    """(F, F(b) - F(a)) con la segunda parte del teorema; ValueError si no aplica."""
    if a == b:
        return sp.Integer(0) * x, sp.Integer(0)
    if not continua_en(f, a, b):
        raise ValueError("f no es continua en todo el intervalo [a, b]: el teorema fundamental no aplica "
                         "y F(b) - F(a) puede dar un número sin sentido.")
    an, bn = sp.nsimplify(a), sp.nsimplify(b)
    if f.has(sp.Abs):                          # |u|: se separa donde u cambia de signo
        return None, _evaluar_por_tramos(f, an, bn)
    F = antiderivada(f)
    return F, sp.simplify(F.subs(x, bn) - F.subs(x, an))


def ceros_en(f, a, b):
    """Ceros de f en (a, b): exactos si sympy los encuentra; si no, por cambio de signo."""
    sol = sp.solveset(f, x, sp.Interval.open(sp.nsimplify(a), sp.nsimplify(b)))
    if isinstance(sol, sp.FiniteSet):
        return sorted(sol, key=float)
    g = numerica(f); xs = np.linspace(float(a), float(b), 2001); y = g(xs)
    return [sp.Float(brentq(lambda s: float(g(np.array([s]))[0]), p, q), 12)
            for p, q, yp, yq in zip(xs, xs[1:], y, y[1:]) if yp * yq < 0]


def _evaluar_por_tramos(f, a, b):
    """∫_a^b de f con valores absolutos: en cada tramo, |u| se sustituye por u o por -u."""
    lo, hi, signo = (a, b, 1) if a < b else (b, a, -1)
    cortes = {lo, hi}
    for u in [e.args[0] for e in f.atoms(sp.Abs)]:
        cortes |= set(ceros_en(u, lo, hi))
    cortes = sorted(cortes, key=float)
    total = 0
    for p, q in zip(cortes, cortes[1:]):
        m = (p + q) / 2
        g = f.replace(sp.Abs, lambda u: u if float(u.subs(x, m)) >= 0 else -u)
        G = antiderivada(g)
        total += G.subs(x, q) - G.subs(x, p)
    return sp.simplify(signo * total)


def punto_medio(f, a, b, m=1000):
    xs = np.linspace(a, b, m + 1)
    return float(numerica(f)((xs[:-1] + xs[1:]) / 2).sum() * (b - a) / m)


def calculadora_evaluacion(f_txt, a, b, n_cifras):
    try:
        f = parsear(f_txt)
        F, valor = evaluar(f, a, b)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"F(x) = {F}" if F is not None else "f tiene valores absolutos: se integró por tramos")
    print(f"[F(x)] de {a} a {b} = {valor} ≈ {cifras(valor, n_cifras)}   ({n_cifras} cifras significativas, redondeado)")
    print(f"comprobación con M_1000 ≈ {cifras(punto_medio(f, a, b), n_cifras)}")


widgets.interact(calculadora_evaluacion,
    f_txt=widgets.Text(value="x**2", description="f(x) ="),
    a=widgets.FloatText(value=0, description="a"), b=widgets.FloatText(value=3, description="b"),
    n_cifras=widgets.IntSlider(value=6, min=1, max=10, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
PRUEBAS_2 = [
    ("∫_0^π sen x dx = 2", lambda: evaluar(sp.sin(x), 0, sp.pi)[1] == 2),
    ("∫_1^e dx/x = 1", lambda: evaluar(1 / x, 1, sp.E)[1] == 1),
    ("∫_0^1 e^(2x) dx = (e^2 - 1)/2", lambda: sp.simplify(evaluar(sp.exp(2 * x), 0, 1)[1] - (sp.E**2 - 1) / 2) == 0),
    ("1/x^2 en [-1, 1] se rechaza (no continua)", lambda: _rechaza(lambda: evaluar(1 / x**2, -1, 1))),
    ("e^(-x^2) en [0, 1]: sin antiderivada elemental, se avisa", lambda: _rechaza(lambda: evaluar(sp.exp(-x**2), 0, 1))),
    ("|x - 1| en [0, 3]: 5/2, por tramos", lambda: evaluar(sp.Abs(x - 1), 0, 3)[1] == sp.Rational(5, 2)),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Calcula a mano $\int_1^4(2x-1)\,dx$.

In [ ]:
mi_valor = None     # escribe un número

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    F, ref = evaluar(2 * x - 1, 1, 4)
    print("La calculadora da: F(x) =", F, "  valor =", ref)
    print("coinciden" if cerca(mi_valor, ref, 1e-9) else "NO coinciden: F(x) = x^2 - x; evalúa en 4 y en 1 y resta en ese orden")

## 3. Integral con signo y área total

$$\text{integral con signo}=\int_a^bf(x)\,dx,\qquad \text{área total}=\int_a^b|f(x)|\,dx=\sum_k\Big|\int_{c_{k}}^{c_{k+1}}f(x)\,dx\Big|,$$

con $c_k$ los ceros de $f$ dentro de $(a,b)$. La calculadora encuentra los ceros, integra cada región con el teorema fundamental y entrega las dos cantidades. Decide tú cuál pide el problema.

In [ ]:
def integral_y_area(f, a, b):
    """(integral con signo, área total, cortes, integrales por región)."""
    if not a < b:
        raise ValueError("el intervalo debe cumplir a < b.")
    cortes = [sp.nsimplify(a)] + ceros_en(f, a, b) + [sp.nsimplify(b)]
    partes = [evaluar(f, p, q)[1] for p, q in zip(cortes, cortes[1:])]
    return sp.nsimplify(sum(partes)), sp.nsimplify(sum(abs(v) for v in partes)), cortes, partes


def calculadora_area(f_txt, a, b, n_cifras):
    try:
        f = parsear(f_txt)
        integral, area, cortes, partes = integral_y_area(f, a, b)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print("ceros dentro del intervalo:", ", ".join(cifras(c, n_cifras) for c in cortes[1:-1]) or "ninguno")
    for p, q, v in zip(cortes, cortes[1:], partes):
        exacto = f"{v} ≈ " if len(str(v)) < 30 else ""
        print(f"  ∫ de {cifras(p, n_cifras)} a {cifras(q, n_cifras)} = {exacto}{cifras(v, n_cifras)}")
    print(f"integral con signo ≈ {cifras(integral, n_cifras)}   área total ≈ {cifras(area, n_cifras)}"
          f"   ({n_cifras} cifras significativas)")
    xs = np.linspace(a, b, 400); y = numerica(f)(xs)
    fig, ax = plt.subplots(figsize=(5.5, 3))
    ax.fill_between(xs, 0, y, where=y >= 0, color="0.8", label="cuenta positiva")
    ax.fill_between(xs, 0, y, where=y <= 0, facecolor="white", edgecolor="black", hatch="///", label="cuenta negativa")
    ax.plot(xs, y, color="navy"); ax.axhline(0, color="black", lw=0.6); ax.legend(); plt.show()


widgets.interact(calculadora_area,
    f_txt=widgets.Text(value="sin(x)", description="f(x) ="),
    a=widgets.FloatText(value=0, description="a"), b=widgets.FloatText(value=4.712389, description="b"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=10, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
PRUEBAS_3 = [
    ("sen x en [0, 3π/2]: integral 1 y área 3",
     lambda: integral_y_area(sp.sin(x), 0, 3 * sp.pi / 2)[:2] == (1, 3)),
    ("x^3 en [-1, 2]: integral 15/4 y área 17/4",
     lambda: integral_y_area(x**3, -1, 2)[:2] == (sp.Rational(15, 4), sp.Rational(17, 4))),
    ("constante 2 en [0, 3]: 6 y 6", lambda: integral_y_area(2 + 0 * x, 0, 3)[:2] == (6, 6)),
    ("cos x en [0, 2π]: integral 0 y área 4", lambda: integral_y_area(sp.cos(x), 0, 2 * sp.pi)[:2] == (0, 4)),
    ("a >= b se rechaza", lambda: _rechaza(lambda: integral_y_area(x, 2, 1))),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Calcula a mano $\int_0^2(x-1)\,dx$ y el área total entre $y=x-1$ y el eje en $[0,2]$.

In [ ]:
mi_integral = None     # escribe un número
mi_area = None         # escribe un número

if mi_integral is None or mi_area is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    integral, area, *_ = integral_y_area(x - 1, 0, 2)
    print("La calculadora da: integral =", integral, "  área =", area)
    print("coinciden" if cerca(mi_integral, integral) and cerca(mi_area, area) else
          "NO coinciden: los dos triángulos miden 1/2; con signo se restan y en el área se suman")

## 4. Valor medio de una función

$$\bar f=\frac{1}{b-a}\int_a^bf(x)\,dx,\qquad f(c)=\bar f\ \text{para algún } c\in[a,b]\ \text{si } f \text{ es continua}.$$

La calculadora da $\bar f$, busca **todos** los $c$ de $[a,b]$ con $f(c)=\bar f$ y dibuja el rectángulo de altura $\bar f$. También muestra el promedio de los extremos, $\frac{f(a)+f(b)}{2}$, que coincide con $\bar f$ siempre que $f$ es lineal y, en otros casos, solo por casualidad.

In [ ]:
def valor_medio(f, a, b):
    """(valor medio exacto, lista de c en [a, b] o 'todo el intervalo')."""
    if not a < b:
        raise ValueError("el intervalo debe cumplir a < b.")
    fbar = sp.simplify(evaluar(f, a, b)[1] / (sp.nsimplify(b) - sp.nsimplify(a)))
    escala = max(1.0, float(np.max(np.abs(numerica(f)(np.linspace(float(a), float(b), 401))))))
    if abs(float(fbar)) < 1e-12 * escala:      # ruido de redondeo: el valor medio es cero
        fbar = sp.Integer(0)
    if sp.simplify(f - fbar) == 0:
        return fbar, "todo el intervalo"
    g = numerica(f - fbar)
    xs = np.linspace(float(a), float(b), 4001); y = g(xs)
    cs = [brentq(lambda s: float(g(np.array([s]))[0]), p, q) for p, q, yp, yq in zip(xs, xs[1:], y, y[1:]) if yp * yq < 0]
    cs += [float(p) for p, yp in zip(xs, y) if abs(yp) < 1e-12 * escala]     # incluye los extremos
    return fbar, sorted(set(round(c, 9) for c in cs))


def calculadora_valor_medio(f_txt, a, b, n_cifras):
    try:
        f = parsear(f_txt)
        fbar, cs = valor_medio(f, a, b)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"valor medio = {fbar} ≈ {cifras(fbar, n_cifras)}   ({n_cifras} cifras significativas, redondeado)")
    print("puntos c:", cs if isinstance(cs, str) else ", ".join(cifras(c, n_cifras) for c in cs))
    extremos = (valor_real(f, a) + valor_real(f, b)) / 2
    print(f"promedio de los extremos ≈ {cifras(0 if abs(extremos) < 1e-12 else extremos, n_cifras)}")
    xs = np.linspace(a, b, 400)
    fig, ax = plt.subplots(figsize=(5.5, 3))
    ax.fill_between([a, b], 0, [float(fbar)] * 2, color="0.85", label="rectángulo de altura f̄")
    ax.plot(xs, numerica(f)(xs), color="navy", label="f(x)")
    ax.axhline(float(fbar), color="red", ls="--")
    if not isinstance(cs, str):
        ax.plot(cs, [float(fbar)] * len(cs), "o", color="red")
    ax.legend(); plt.show()


widgets.interact(calculadora_valor_medio,
    f_txt=widgets.Text(value="2 + sin(x)", description="f(x) ="),
    a=widgets.FloatText(value=0, description="a"), b=widgets.FloatText(value=4, description="b"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=10, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
PRUEBAS_4 = [
    ("x^2 en [0, 3]: valor medio 3, c = √3",
     lambda: valor_medio(x**2, 0, 3)[0] == 3 and cerca(valor_medio(x**2, 0, 3)[1][0], math.sqrt(3), 1e-9)),
    ("sen x en [0, π]: 2/π y dos puntos c",
     lambda: valor_medio(sp.sin(x), 0, sp.pi)[0] == 2 / sp.pi and len(valor_medio(sp.sin(x), 0, sp.pi)[1]) == 2),
    ("3x en [0, 2]: valor medio 3 en c = 1", lambda: valor_medio(3 * x, 0, 2)[0] == 3 and cerca(valor_medio(3 * x, 0, 2)[1][0], 1)),
    ("constante 5 en [1, 4]: todo el intervalo", lambda: valor_medio(5 + 0 * x, 1, 4) == (5, "todo el intervalo")),
    ("2 + sen x en [0, 4]: c ≈ 0.4262 y 2.7154",
     lambda: [round(c, 4) for c in valor_medio(2 + sp.sin(x), 0, 4)[1]] == [0.4262, 2.7154]),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Calcula a mano el valor medio de $f(x)=2x+1$ en $[0,4]$ y el punto $c$.

In [ ]:
mi_media = None   # escribe un número
mi_c = None       # escribe un número

if mi_media is None or mi_c is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    fbar, cs = valor_medio(2 * x + 1, 0, 4)
    print("La calculadora da: valor medio =", fbar, "  c =", cs)
    print("coinciden" if cerca(mi_media, fbar) and cerca(mi_c, cs[0]) else
          "NO coinciden: la integral vale 20 y el intervalo mide 4; con f lineal, c es el punto medio")

## 5. Comprobación numérica contra la suma de Riemann

Si $F$ es correcta y $f$ continua, las diferencias entre $M_n$, $T_n$, $S_n$ y $F(b)-F(a)$ tienden a cero al crecer $n$. Si se **estancan**, $F$ está mal. Si las sumas **crecen sin cota**, la integral no existe como número finito.

El comprobador recibe $f$ y, si quieres, tu antiderivada $F$ (si la dejas vacía usa la de `sympy`, o solo las sumas si no hay antiderivada elemental). Da la tabla para $n=2,4,\dots,256$ y un veredicto.

In [ ]:
_NS = (2, 4, 8, 16, 32, 64, 128, 256)


def sumas(f, a, b, n):
    """(M_n, T_n, S_n) de f en [a, b]; n par."""
    g = numerica(f); xs = np.linspace(a, b, n + 1); y = g(xs); h = (b - a) / n
    M = float(g((xs[:-1] + xs[1:]) / 2).sum() * h)
    T = float(h * (y.sum() - (y[0] + y[-1]) / 2))
    S = float(h / 3 * (y[0] + y[-1] + 4 * y[1:-1:2].sum() + 2 * y[2:-1:2].sum()))
    return M, T, S


def comprobar(f, a, b, F=None):
    """(veredicto, valor de referencia o None, filas (n, M, T, S))."""
    if not a < b:
        raise ValueError("el intervalo debe cumplir a < b.")
    if not continua_en(f, a, b):
        g = numerica(f)
        Ms = []
        for n in (10, 100, 1000):
            xs = np.linspace(a, b, n + 1)
            with np.errstate(all="ignore"):
                Ms.append(float(g((xs[:-1] + xs[1:]) / 2).sum() * (b - a) / n))
        A = [abs(m) for m in Ms]
        crece = all(np.isfinite(Ms)) and A[2] > 3 * A[1] > 9 * A[0] > 0
        motivo = "f no es continua en [a, b] y las sumas crecen sin cota" if crece else "f no es continua en [a, b]"
        return f"no aplica: {motivo}; la integral no existe como número finito o el teorema no se puede usar", None, \
               [(n, m, float("nan"), float("nan")) for n, m in zip((10, 100, 1000), Ms)]
    filas = [(n,) + sumas(f, a, b, n) for n in _NS]
    if F is None:
        try:
            F = antiderivada(f)
        except ValueError:
            return "sin antiderivada elemental: el valor es el que dan las sumas", filas[-1][3], filas
    try:
        valor = valor_real(F, b) - valor_real(F, a)
    except (ValueError, TypeError) as err:
        raise ValueError("tu F no tiene valor real en los extremos del intervalo.") from err
    d = abs(filas[-1][3] - valor)                      # S_256 contra F(b) - F(a)
    e = abs(filas[-1][3] - filas[-2][3])               # cuánto cambió S_n en la última duplicación
    tol = 1e-9 * max(1, abs(valor))
    if d <= 10 * e + tol:
        return "coinciden", valor, filas
    if d > 100 * e + tol:
        return "no coinciden: revisa F", valor, filas
    return "no concluyente: las sumas aún no se estabilizan; aumenta n", valor, filas


def calculadora_comprobacion(f_txt, F_txt, a, b):
    try:
        f = parsear(f_txt)
        F = parsear(F_txt) if F_txt.strip() else None
        veredicto, valor, filas = comprobar(f, a, b, F)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    if valor is not None:
        print(f"valor de referencia: {cifras(valor, 8)}   (8 cifras significativas)")
    print("    n          M_n          T_n          S_n")
    for n, M, T, S in filas:
        celdas = [cifras(v, 8) if math.isfinite(v) else "—" for v in (M, T, S)]
        print(f"{n:5d}  {celdas[0]:>11}  {celdas[1]:>11}  {celdas[2]:>11}")
    print("Veredicto:", veredicto)


widgets.interact(calculadora_comprobacion,
    f_txt=widgets.Text(value="cos(2*x)", description="f(x) ="),
    F_txt=widgets.Text(value="sin(2*x)", description="tu F(x) ="),
    a=widgets.FloatText(value=0, description="a"), b=widgets.FloatText(value=1, description="b"));

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
PRUEBAS_5 = [
    ("cos 2x con F = sen(2x)/2: coinciden", lambda: comprobar(sp.cos(2 * x), 0, 1, sp.sin(2 * x) / 2)[0] == "coinciden"),
    ("cos 2x con F = sen 2x: no coinciden", lambda: comprobar(sp.cos(2 * x), 0, 1, sp.sin(2 * x))[0].startswith("no coinciden")),
    ("1/x^2 en [-1, 1]: no aplica, las sumas crecen", lambda: "crecen" in comprobar(1 / x**2, -1, 1)[0]),
    ("e^(-x^2) en [0, 1]: S_16 = 0.746824", lambda: round(sumas(sp.exp(-x**2), 0, 1, 16)[2], 6) == 0.746824),
    ("x^2 en [0, 3]: M_4 = 8.859375 (caso del prompt)", lambda: cerca(sumas(x**2, 0, 3, 4)[0], 8.859375)),
    ("sen 50x en [0, 10] con F correcta: coinciden aunque oscile",
     lambda: comprobar(sp.sin(50 * x), 0, 10, -sp.cos(50 * x) / 50)[0] == "coinciden"),
    ("√x en [0, 1] con F correcta: coinciden aunque S_n converja lento",
     lambda: comprobar(sp.sqrt(x), 0, 1, sp.Rational(2, 3) * x**sp.Rational(3, 2))[0].startswith("coinciden")),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 5).** Para $\int_0^1\big(x^3+1\big)dx$, calcula a mano $F(1)-F(0)$ y $M_2$.

In [ ]:
mi_F = None    # escribe F(1) - F(0)
mi_M2 = None   # escribe M_2

if mi_F is None or mi_M2 is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    ref = evaluar(x**3 + 1, 0, 1)[1]; M2 = sumas(x**3 + 1, 0, 1, 2)[0]
    print("La calculadora da: F(1) - F(0) =", ref, "  M_2 =", M2)
    print("coinciden" if cerca(mi_F, ref) and cerca(mi_M2, M2) else
          "NO coinciden: M_2 evalúa x^3 + 1 en 0.25 y 0.75 y multiplica cada valor por 0.5")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")